# FerrisRes — T4 / Vulkan probe

**v2.** v1 answered half the question: `nvidia-smi` sees **2× Tesla T4**
(driver 580.178.04), and the image has `libvulkan.so.1` but **no ICD manifest
directory at all** (`/usr/share/vulkan/icd.d/` absent, `nvidia_icd.json`
missing). A Vulkan loader with no ICD enumerates nothing, so this kernel writes
the NVIDIA manifest itself, points `VK_DRIVER_FILES`/`VK_ICD_FILENAMES` at it,
and then asks `ferrisres devices --json`.

It also prints `/kaggle/input` because v1 found no `.deb` there.

Stops (no CPU fallback) if no hardware adapter is enumerated.


In [ ]:
import glob, json, os, shutil, subprocess
from pathlib import Path

def run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

smi = shutil.which('nvidia-smi')
if not smi:
    raise SystemExit('no nvidia-smi: CPU session. Settings -> Accelerator -> GPU T4 x2, Internet on, Restart Session.')
print(run([smi, '-L']).stdout)
assert 'T4' in run([smi, '-L']).stdout, 'nvidia-smi sees no T4'

print('--- loader ---')
print(run(['bash', '-lc', 'ldconfig -p | grep -i vulkan || echo none']).stdout)
print('--- ICU search ---')
print(run(['bash', '-lc', 'ls -d /usr/share/vulkan /etc/vulkan 2>/dev/null; find / -name "*icd*.json" -not -path "*/proc/*" 2>/dev/null | head -20']).stdout)
print('--- nvidia userspace libraries ---')
print(run(['bash', '-lc', 'ls -l /usr/lib/x86_64-linux-gnu/libGLX_nvidia.so* /usr/lib/x86_64-linux-gnu/libvulkan.so* 2>&1']).stdout)
print('--- kaggle input ---')
print(run(['bash', '-lc', 'ls -la /kaggle/input 2>&1; find /kaggle/input -maxdepth 3 2>/dev/null | head -40']).stdout)

# The image ships the driver but no ICD manifest. Write one and point the loader
# at it; without this a Vulkan loader enumerates zero devices.
MANIFEST = Path('/kaggle/working/nvidia_icd.json')
MANIFEST.write_text(json.dumps({
    'file_format_version': '1.0.0',
    'ICD': {'library_path': 'libGLX_nvidia.so.0', 'api_version': '1.3.260'},
}, indent=2))
for var in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES'):
    os.environ[var] = str(MANIFEST)
os.environ.setdefault('WGPU_BACKEND', 'vulkan')
print('ICD:', MANIFEST, os.environ['VK_DRIVER_FILES'])


In [ ]:
# v2/v3 showed the image has the driver but not its graphics userspace:
# libGLX_nvidia.so.0 absent, no /usr/share/vulkan/icd.d, and wgpu falls back to
# software GL. v3's plain `apt-get install libnvidia-gl-580` reported success but
# left no library, and the -server variant died on a file owned by
# libnvidia-egl-gbm1. Force past that and prove what actually landed.
diag = run(['bash', '-lc', 'cat /proc/driver/nvidia/version | head -2; echo CAPS=$NVIDIA_DRIVER_CAPABILITIES'])
print(diag.stdout)

ver = run(['bash', '-lc', "nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1"]).stdout.strip()
major = ver.split('.')[0]
print('driver', ver, 'major', major)

print(run(['bash', '-lc', 'apt-get update -qq 2>&1 | tail -1']).stdout)

# 1. force the overwrite that killed v3
r = run(['bash', '-lc',
    'DEBIAN_FRONTEND=noninteractive apt-get install -y -qq '
    '-o Dpkg::Options::="--force-overwrite" '
    '-o Dpkg::Options::="--force-confold" '
    f'libnvidia-gl-{major}-server 2>&1 | tail -12'])
print('--- install -server:', r.returncode)
print(r.stdout)

# 2. if apt has no candidate, take the .deb it already cached and force it
if r.returncode != 0:
    cached = run(['bash', '-lc', f'ls /var/cache/apt/archives/libnvidia-gl-{major}*.deb 2>/dev/null'])
    print('cached debs:', cached.stdout)
    for deb in cached.stdout.split():
        rr = run(['bash', '-lc', f'dpkg -i --force-overwrite --force-confold {deb} 2>&1 | tail -6'])
        print('--- dpkg', deb, rr.returncode)
        print(rr.stdout)

# 3. what did the package actually put on disk
print(run(['bash', '-lc',
    f'dpkg -L libnvidia-gl-{major}-server 2>/dev/null | grep -iE "GLX|vulkan|icd|glcore" | head -20; '
    'echo ---; '
    "find / -xdev \( -name 'libGLX_nvidia*' -o -name 'nvidia_icd.json' -o -name 'libnvidia-vulkan*' -o -name 'libnvidia-glcore*' \) 2>/dev/null | head -20"]).stdout)

libs = run(['bash', '-lc',
    "find / -xdev -name 'libGLX_nvidia.so.0' 2>/dev/null | head -1"]).stdout.strip()
icd = run(['bash', '-lc',
    "find / -xdev -name 'nvidia_icd.json' 2>/dev/null | head -1"]).stdout.strip()
print('libGLX:', libs or 'ABSENT')
print('icd   :', icd or 'ABSENT')

if libs:
    MANIFEST.write_text(json.dumps({
        'file_format_version': '1.0.0',
        'ICD': {'library_path': libs, 'api_version': '1.3.260'},
    }, indent=2))
    print('manifest:', MANIFEST.read_text())
elif icd:
    # the package shipped its own manifest; point the loader at it directly
    for var in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES'):
        os.environ[var] = icd
    print('using the package manifest', icd)


In [ ]:
debs = sorted(Path('/kaggle/input').rglob('ferrisres_*_amd64.deb'))
if not debs:
    print('no .deb mounted; fetching sectionme/ferrisres with kagglehub')
    try:
        import kagglehub
    except ImportError:
        subprocess.run(['pip', 'install', '-q', 'kagglehub'], check=False)
        import kagglehub
    local = Path(kagglehub.dataset_download('sectionme/ferrisres'))
    debs = sorted(local.rglob('*.deb'))
assert debs, 'no .deb from the mount or kagglehub'
print('deb:', debs[-1])
print(run(['dpkg', '-i', str(debs[-1])]).stdout[-1500:])
print(run(['bash', '-lc', 'ferrisres --version']).stdout)

BIN = run(['bash', '-lc', 'command -v ferrisres']).stdout.strip()
assert BIN, 'ferrisres not on PATH'

import re as _re
# `devices --json` logs through tracing on stdout, ANSI-coloured. Silence the
# logs and strip escapes, then take the last array in the stream.
probe = run([BIN, 'devices', '--json'], env={**os.environ, 'RUST_LOG': 'error'})
print('devices exit', probe.returncode)
print(probe.stdout[-3000:])
print(probe.stderr[-2000:])
_clean = _re.sub(r'\x1b\[[0-9;]*m', '', probe.stdout)
_idx = _clean.rfind('\n[')
devices = json.JSONDecoder().raw_decode(_clean, _idx + 1)[0] if _idx >= 0 else []
assert isinstance(devices, list) and devices, (
    'devices --json produced no parseable array; first 300 chars: ' + repr(_clean[:300])
)

adapters = devices
hw = [a for a in adapters if not a.get('software', False) and a.get('kind') != 'Cpu']
print('hardware adapters:', [(a.get('index'), a.get('name'), a.get('kind'), a.get('backend')) for a in hw])
assert hw, 'wgpu enumerates no hardware adapter even after the graphics userspace install'
t4 = [a for a in hw if 'T4' in a.get('name', '')]
assert t4, 'a hardware adapter exists but it is not a T4: ' + repr([a['name'] for a in hw])
WGPU_ADAPTER = str(t4[0]['index'])
os.environ['WGPU_ADAPTER_INDEX'] = WGPU_ADAPTER
os.environ['WGPU_BACKEND'] = 'vulkan'
print('selected adapter', WGPU_ADAPTER, t4[0]['name'])


In [ ]:
import os, re, time
from pathlib import Path

corpus = Path('/kaggle/working/probe.txt')
words = 'the quick brown fox jumps over the lazy dog'.split()
lines = []
for i in range(100):
    lines.append(' '.join(words[(i + k) % len(words)] for k in range(len(words))))
    lines.append('the answer is %d .' % (i % 4))
corpus.write_text(chr(10).join(lines) + chr(10))

EPOCHS = 3  # ~58 steps on this corpus, enough to time a step
OUT = Path('/kaggle/working/probe.ckpt')
cmd = [BIN, 'train', '--data', str(corpus), '--data-format', 'text', '--chat-template', 'gemma',
       '--hidden-dim', '64', '--num-blocks', '2', '--block-size', '4',
       '--epochs', str(EPOCHS), '--batch-size', '4', '--learning-rate', '3e-3',
       '--seq-len', '128', '--vocab', '259', '--seed', '42',
       '--checkpoint-every', '1000', '--output', str(OUT), '--log-every', '5',
       '--device', 'gpu']

t0 = time.perf_counter()
proc = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0
log = proc.stdout + proc.stderr
print(log[-6000:])
assert proc.returncode == 0, f'train --device gpu failed ({proc.returncode})'

m = re.search(r'steps=(\d+).*initial_loss=([0-9.]+) final_loss=([0-9.]+)', log)
assert m, 'no training summary'
steps = int(m.group(1))
tokens = steps * 4 * 128
summary = {
    'adapter_index': WGPU_ADAPTER,
    'adapter_name': t4[0]['name'],
    'steps': steps,
    'seconds': round(elapsed, 2),
    'tokens_per_s': round(tokens / elapsed, 1),
    'initial_loss': float(m.group(2)),
    'final_loss': float(m.group(3)),
    'checkpoint': str(OUT),
    'checkpoint_bytes': OUT.stat().st_size if OUT.exists() else None,
}
Path('/kaggle/working/probe.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
assert summary['final_loss'] < summary['initial_loss'], 'loss did not fall: the GPU path ran but did not learn'
